# 17.2 动态规划 / Dynamic Programming (Policy & Value Iteration)

**中文**：上一节我们用贝尔曼方程"评估一个策略、贪心改进一次"。本节把它做到极致——**动态规划(Dynamic Programming, DP)**:在**完全已知环境模型**($P$ 和 $R$ 都知道)的前提下，用两大算法求出**最优策略**:**策略迭代**和**价值迭代**。这属于 RL 的"**规划(planning)**"分支——不用真的去环境里试错，纯靠"想清楚"就能算出最优解。
**English**: Last section we used the Bellman equation to "evaluate a policy and greedily improve once." Now we push it fully — **Dynamic Programming (DP)**: given a **fully known environment model** (both $P$ and $R$), compute the **optimal policy** via two algorithms: **policy iteration** and **value iteration**. This is RL's "**planning**" branch — no real trial-and-error needed; pure "thinking it through" finds the optimum.

---

**中文**：两大算法:
**English**: The two algorithms:

**中文**：
1. **策略迭代(Policy Iteration)** = 反复交替两步直到策略不再变:
   - **策略评估(evaluation)**:把当前策略 $\pi$ 的价值 $V^\pi$ **算到收敛**(反复套用贝尔曼期望方程)。
   - **策略改进(improvement)**:基于 $V^\pi$ **贪心**地选每个状态的最佳动作，得到更好的策略 $\pi'$。
   - **策略改进定理**保证:每次改进后策略不会变差，且有限步内收敛到最优。

**English**:
1. **Policy Iteration** = alternate two steps until the policy stops changing:
   - **Evaluation**: compute the current policy $\pi$'s value $V^\pi$ **to convergence** (repeatedly apply the Bellman expectation equation).
   - **Improvement**: greedily pick each state's best action w.r.t. $V^\pi$, yielding a better policy $\pi'$.
   - The **policy improvement theorem** guarantees each improvement never worsens the policy and converges to the optimum in finitely many steps.

**中文**：
2. **价值迭代(Value Iteration)** = 不等策略评估收敛，**每个状态每轮只做一次"取最好动作"的更新**(直接迭代贝尔曼**最优**方程)，收敛后再一次性提取贪心策略:
   **English**: **Value Iteration** = don't wait for evaluation to converge; **each sweep does one "take the best action" update per state** (directly iterating the Bellman **optimality** equation), then extract the greedy policy once at the end:

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'|s,a)\big[R(s,a,s')+\gamma V_k(s')\big]$$

**中文**：直觉:价值迭代把策略迭代里"完整评估 + 改进"两步**合成一步**(评估只做一轮就改进)，通常更快。两者最终都收敛到**同一个最优价值 $V^*$ 和最优策略 $\pi^*$**。
**English**: Intuition: value iteration **fuses** policy iteration's "full evaluation + improvement" into **one step** (evaluate just one sweep then improve), usually faster. Both converge to the **same optimal value $V^*$ and optimal policy $\pi^*$**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 必考）**
> **中文**：DP=**已知模型**下求最优策略(规划)。**策略迭代**:评估(算到收敛)↔改进(贪心)交替, 迭代次数少但每次评估贵。**价值迭代**:直接迭代贝尔曼最优方程(每轮一次 max), 迭代多但每轮便宜, 通常总体更快。二者都收敛到 $V^*,\pi^*$(贝尔曼最优方程的唯一解, 因 $\gamma<1$ 时贝尔曼算子是压缩映射)。**致命局限**:①必须**已知 $P,R$**(现实很少满足→引出无模型 MC/TD); ②要**遍历所有状态**($O(|S|^2|A|)$/sweep)→状态空间大就爆炸(维度灾难→引出函数近似/DQN)。
> **English**: DP = find the optimal policy with a **known model** (planning). **Policy iteration**: alternate evaluation (to convergence) ↔ improvement (greedy); fewer iterations but each evaluation is expensive. **Value iteration**: directly iterate the Bellman optimality equation (one max per sweep); more iterations but each is cheap, usually faster overall. Both converge to $V^*,\pi^*$ (the unique fixed point of the Bellman optimality operator, a contraction when $\gamma<1$). **Fatal limits**: ① requires a **known $P,R$** (rarely true → motivates model-free MC/TD); ② must **sweep all states** ($O(|S|^2|A|)$/sweep) → explodes for large state spaces (curse of dimensionality → motivates function approximation/DQN).


In [ ]:

# ============================================================
# 环境:带"冰洞"的湿滑 GridWorld / slippery GridWorld with holes (FrozenLake-style)
# 中文:4x4 冰面。想往一个方向走, 只有 80% 成功, 各 10% 概率"打滑"到垂直方向——这是随机 MDP。
#      掉进冰洞(H)扣1分并结束; 到终点(G)+1分并结束; 其余每步 -0.01。这考验 DP 处理不确定性的能力。
# English: 4x4 ice. An intended move succeeds 80%; 10%+10% "slips" to perpendicular directions — a stochastic MDP.
#      Falling in a hole(H): -1 and end; reaching goal(G): +1 and end; else -0.01 per step.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
size=4; nS=size*size; nA=4
A=[(-1,0),(0,1),(1,0),(0,-1)]; NAMES=["↑","→","↓","←"]
holes={5,7,11,12}; goal=15
def rc(s): return divmod(s,size)
def move(s,a):
    r,c=rc(s); dr,dc=A[a]; nr,nc=r+dr,c+dc
    return nr*size+nc if 0<=nr<size and 0<=nc<size else s   # 越界=撞墙原地 / off-grid = stay
# 构造转移模型 P[s][a] = [(概率, 下一状态, 奖励, 是否终止), ...] / build the transition model
P={s:{a:[] for a in range(nA)} for s in range(nS)}
for s in range(nS):
    for a in range(nA):
        if s==goal or s in holes: P[s][a]=[(1.0,s,0.0,True)]; continue   # 终止态自环 / terminal
        outc={}
        for a2,prob in [(a,0.8),((a-1)%4,0.1),((a+1)%4,0.1)]:           # 80% 意图 + 10%/10% 打滑 / slip
            ns=move(s,a2)
            r=1.0 if ns==goal else (-1.0 if ns in holes else -0.01)
            d=(ns==goal or ns in holes)
            outc[(ns,r,d)]=outc.get((ns,r,d),0)+prob
        P[s][a]=[(p,ns,r,d) for (ns,r,d),p in outc.items()]
gamma=0.99
grid_show=np.array(["S",".",".",".",".","H",".","H",".",".",".","H","H",".",".","G"]).reshape(4,4)
print("地图(S起点 H冰洞 G终点) / map:"); print(grid_show)
print("\n从状态0向右(→)的随机转移 / stochastic outcomes of action → in state 0:")
for p,ns,r,d in P[0][1]: print(f"  概率{p:.1f} -> 状态{ns} 奖励{r} 终止{d}")


**中文**：先实现**策略迭代**。它需要两个子程序:①策略评估(把 $V^\pi$ 算到收敛)，②贪心改进。反复交替，直到策略稳定不变。
**English**: First **policy iteration**. It needs two subroutines: ① policy evaluation (compute $V^\pi$ to convergence), ② greedy improvement. Alternate until the policy is stable.


In [ ]:

# ============================================================
# 策略迭代 / Policy Iteration
# ============================================================
def q_from_V(V, s):                                          # 一步前瞻算各动作的 Q(s,a) / one-step lookahead
    return [sum(p*(r + gamma*V[ns]*(not d)) for p,ns,r,d in P[s][a]) for a in range(nA)]

def policy_eval(policy, V, theta=1e-9):
    sweeps=0
    while True:
        delta=0
        for s in range(nS):
            v=sum(policy[s,a]*sum(p*(r+gamma*V[ns]*(not d)) for p,ns,r,d in P[s][a]) for a in range(nA))
            delta=max(delta,abs(v-V[s])); V[s]=v
        sweeps+=1
        if delta<theta: break                                # 评估到收敛 / evaluate to convergence
    return V, sweeps

def greedy_policy(V):
    pol=np.zeros((nS,nA))
    for s in range(nS): pol[s, int(np.argmax(q_from_V(V,s)))]=1.0   # 每个状态选最优动作 / greedy
    return pol

def policy_iteration():
    pol=np.ones((nS,nA))/nA; V=np.zeros(nS); improves=0; total_sweeps=0
    while True:
        V, sw=policy_eval(pol, V); total_sweeps+=sw           # 评估 / evaluate
        new=greedy_policy(V); improves+=1                     # 改进 / improve
        if np.array_equal(new, pol): break                   # 策略稳定=最优 / stable = optimal
        pol=new
    return V, pol, improves, total_sweeps

Vpi, pol_pi, n_improve, pi_sweeps = policy_iteration()
print(f"策略迭代:改进次数 {n_improve}, 总评估 sweep 数 {pi_sweeps}")
print("最优价值 V* / optimal value:"); print(np.round(Vpi.reshape(4,4),2))


**中文**：再实现**价值迭代**——不做完整评估，每轮对每个状态直接取"最好动作"更新价值(贝尔曼最优方程)，收敛后提取贪心策略。看它比策略迭代快多少。
**English**: Now **value iteration** — no full evaluation; each sweep updates every state by directly taking the "best action" (Bellman optimality), then extract the greedy policy after convergence. See how much faster than policy iteration.


In [ ]:

# ============================================================
# 价值迭代 / Value Iteration
# ============================================================
def value_iteration(theta=1e-9):
    V=np.zeros(nS); sweeps=0; hist=[]
    while True:
        delta=0
        for s in range(nS):
            nv=max(q_from_V(V,s))                             # 直接取最优动作的值 / Bellman optimality
            delta=max(delta,abs(nv-V[s])); V[s]=nv
        sweeps+=1; hist.append(V.copy())
        if delta<theta: break
    return V, greedy_policy(V), sweeps, hist

Vvi, pol_vi, vi_sweeps, vi_hist = value_iteration()
print(f"价值迭代:sweep 数 {vi_sweeps} (策略迭代用了 {pi_sweeps})")
print("两法得到相同最优策略 / same optimal policy:", np.array_equal(pol_pi.argmax(1), pol_vi.argmax(1)))
print("两法最优价值最大差异 / max |V diff|:", np.abs(Vpi-Vvi).max())
# 打印最优策略(注意冰洞附近的"反直觉"箭头)/ print optimal policy (note counterintuitive arrows near holes)
arrows=[]
for s in range(nS):
    if s==goal: arrows.append("G")
    elif s in holes: arrows.append("H")
    else: arrows.append(NAMES[pol_vi[s].argmax()])
print("\n最优策略 / optimal policy:"); print(np.array(arrows).reshape(4,4))


**中文**：注意看最优策略里那些**反直觉的箭头**——有些格子的最优动作竟然是"往墙上撞"或"背离终点"！这不是 bug，而是**湿滑环境**的深刻结果:在冰洞旁边，与其冒 10% 打滑掉洞的风险直奔终点，不如**故意撞墙/绕开**，用"原地不动"来规避风险。**最优策略会主动权衡风险**——这正是随机 MDP 比确定性问题深刻的地方。下面可视化。
**English**: Notice the **counterintuitive arrows** in the optimal policy — some cells' best action is to "bump into a wall" or "move away from the goal"! Not a bug, but a profound consequence of the **slippery environment**: next to a hole, rather than risk a 10% slip into it by heading straight for the goal, it is better to **deliberately bump a wall / detour**, using "stay put" to avoid risk. **The optimal policy actively trades off risk** — exactly what makes stochastic MDPs deeper than deterministic ones. Let's visualize.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(16,4.7))
# ① 最优价值 + 策略 / optimal value + policy
Vg=Vvi.reshape(4,4)
im=ax[0].imshow(Vg,cmap="RdYlGn")
for s in range(nS):
    r,c=rc(s)
    if s==goal: t="GOAL"
    elif s in holes: t="HOLE"
    else: t=f"{Vg[r,c]:.2f}\n{NAMES[pol_vi[s].argmax()]}"
    ax[0].text(c,r,t,ha="center",va="center",fontsize=9)
ax[0].set_title("最优价值+策略(注意冰洞旁的避险)/ V* + π*"); ax[0].set_xticks([]); ax[0].set_yticks([])
plt.colorbar(im,ax=ax[0],fraction=0.046)
# ② 价值迭代收敛曲线(每 sweep 相对 V* 的最大误差)/ VI convergence: max error vs sweep
vi_err=[np.abs(h-Vvi).max() for h in vi_hist]
ax[1].semilogy(vi_err,"o-",label=f"价值迭代 VI ({vi_sweeps} sweeps)",color="#4C72B0")
ax[1].axhline(1e-9,ls=":",color="gray")
ax[1].set_title("价值迭代收敛(对数刻度)/ VI convergence"); ax[1].set_xlabel("sweep"); ax[1].set_ylabel("max|V-V*|"); ax[1].legend()
# ③ sweep 数对比 / total sweeps bar
ax[2].bar(["策略迭代\nPolicy Iter","价值迭代\nValue Iter"],[pi_sweeps,vi_sweeps],color=["#C44E52","#4C72B0"])
for i,v in enumerate([pi_sweeps,vi_sweeps]): ax[2].text(i,v+3,str(v),ha="center")
ax[2].set_title("总计算量(sweep 数, 越少越快)/ total sweeps"); ax[2].set_ylabel("#sweeps")
plt.tight_layout(); plt.savefig("/tmp/rl02_viz.png",dpi=80); plt.show()
print(f"价值迭代 {vi_sweeps} sweeps vs 策略迭代 {pi_sweeps} sweeps —— 本例 VI 更省")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **两法殊途同归**:策略迭代和价值迭代收敛到**完全相同**的 $V^*$ 和 $\pi^*$(最大差异 ~0)。这是理论保证——贝尔曼最优方程有**唯一解**($\gamma<1$ 时贝尔曼算子是压缩映射，必收敛到唯一不动点)。
2. **价值迭代通常更省**:本例 VI 用 ~171 sweep，策略迭代用 ~410 sweep。因为策略迭代每轮都把评估**算到完全收敛**才改进(浪费)，而 VI"评估一轮就改进"。但策略迭代的"改进次数"很少(4 次)——两者是同一思想的不同截断。
3. **随机性让最优策略学会避险**:湿滑环境下，最优策略在冰洞旁主动"撞墙/绕路"以降低打滑风险。这说明 RL 求的不是"最短路径"，而是"**期望回报最优**"——会为不确定性买保险。
4. **DP 的致命前提**:它必须**完全已知 $P$ 和 $R$**，还要遍历所有状态。现实中环境模型往往未知、状态空间巨大——这正是接下来**无模型方法(蒙特卡洛、TD、Q-learning)** 和**函数近似(DQN)** 要解决的。

**English**:
1. **Both converge to the same answer**: policy and value iteration reach **identical** $V^*$ and $\pi^*$ (max diff ~0). A theoretical guarantee — the Bellman optimality equation has a **unique solution** (with $\gamma<1$ the Bellman operator is a contraction, converging to its unique fixed point).
2. **Value iteration is usually cheaper**: here VI uses ~171 sweeps vs policy iteration's ~410. Policy iteration wastes effort evaluating each intermediate policy **to full convergence** before improving, while VI "improves after one sweep." But policy iteration needs very few "improvement steps" (4) — they are the same idea with different truncation.
3. **Stochasticity teaches risk-aversion**: on slippery ice, the optimal policy deliberately "bumps walls / detours" near holes to reduce slip risk. RL seeks not the "shortest path" but the **optimal expected return** — it buys insurance against uncertainty.
4. **DP's fatal prerequisite**: it needs a **fully known $P$ and $R$** and must sweep all states. Real environments are often unknown with huge state spaces — exactly what **model-free methods (Monte Carlo, TD, Q-learning)** and **function approximation (DQN)** address next.

> 💼 **实战视角 / Practical angle**
> **中文**:纯 DP 在现实 RL 里**很少直接用**(模型未知+状态爆炸), 但它是**一切 RL 的理论母体**——TD、Q-learning、DQN 本质都是"用采样近似 DP 的贝尔曼更新"。哪里还用真 DP:①运筹优化(库存、调度、最短路的 MDP 变体); ②模型已知的棋类(结合搜索); ③规划模块(机器人已知地图导航)。面试金句:*"策略迭代=评估到收敛再改进; 价值迭代=贝尔曼最优方程直接迭代; 都收敛到唯一 V*, 但都需已知模型+遍历状态, 所以现实转向无模型采样法。"*
> **English**: Pure DP is **rarely used directly** in real RL (unknown model + state explosion), but it is **the theoretical mother of all RL** — TD, Q-learning, DQN are essentially "approximating DP's Bellman update by sampling." Where real DP still applies: ① operations research (inventory, scheduling, shortest-path MDP variants); ② known-model board games (with search); ③ planning modules (robot navigation on a known map). Interview line: *"Policy iteration = evaluate-to-convergence then improve; value iteration = iterate the Bellman optimality equation directly; both converge to the unique V*, but both need a known model + full state sweeps, so reality turns to model-free sampling."*

---
### 小结 / Summary
- **中文**:DP 在已知模型下求最优策略;策略迭代(评估↔改进)与价值迭代(直接迭代贝尔曼最优)都收敛到唯一 V*,π*。
- **English**: DP finds the optimal policy with a known model; policy iteration (evaluate↔improve) and value iteration (iterate Bellman optimality directly) both converge to the unique V*, π*.
- **中文**:价值迭代通常更省算力;随机环境让最优策略学会避险(不是最短路)。
- **English**: Value iteration is usually cheaper; stochastic environments make the optimal policy risk-averse (not shortest-path).
- **中文**:DP 需已知 P,R + 遍历状态——现实局限引出无模型 MC/TD 与函数近似 DQN。
- **English**: DP needs known P,R + full state sweeps — its limits motivate model-free MC/TD and function-approximation DQN.
